# 🎯 [K10_M08] 생성형 AI용 시스템 프롬프트 만들기
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K10/k10_m08_solution.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 병영생활·상담 (Level 1) / K10 군 생활 정보 챗봇 |
| **미션** | 미션 08. 생성형 AI용 시스템 프롬프트 만들기 (소요시간: 약 2.0시간) |
| **실습 유형** | 자연어 처리(NLP) 및 군 생활 안내 대화형 챗봇 구축 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.



In [1]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


실행 환경 준비 완료 (Python 3.14.10)


---

## 🎯 미션 08. 생성형 AI용 시스템 프롬프트 만들기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **FAQ와 대화 규칙을 담은 생성형 AI용 시스템 프롬프트를 자동으로 만듭니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **역할, 규칙 5개, FAQ 전체(주제·담당·답변·질문 예시), 모를 때 문구를 담은 시스템 프롬프트와 분류기 힌트가 붙은 사용자 메시지를 생성 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 3개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `시스템 프롬프트 생성 함수, 생성 예시, 점검 결과` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 프롬프트 설계 — 규칙과 FAQ를 담은 프롬프트를 만들고 누락을 점검했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(시스템 프롬프트 생성 함수, 생성 예시, 점검 결과)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [9]:
"""FAQ와 대화 규칙을 담아 생성형 AI에 넣을 시스템 프롬프트와 질문별 사용자 메시지를 자동으로 만들고 필수 요소가 빠지지 않았는지 점검한다."""
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 전입 병사 생활 FAQ (실제 부대·규정·인물과 무관)
# 의도 = (코드, 이름, 담당 부서, 답변, 질문 예시 6개). 앞 4개는 학습용, 뒤 2개는 시험용으로 쓴다.
# ---------------------------------------------------------------------------
FAQ = [
    ("schedule", "일과 시간", "중대 행정반",
     "기상은 06시 30분, 점호는 21시 30분, 취침은 22시입니다. 평일 18시부터 21시까지는 개인정비 시간입니다.",
     ["하루 일과는 어떻게 되나요", "기상 시간이 몇 시예요", "점호는 몇 시에 하나요", "취침 시간 알려주세요",
      "개인정비 시간은 언제인가요", "아침에 몇 시에 일어나야 해요"]),
    ("leave", "휴가", "중대 행정반",
     "정기 휴가는 시행 7일 전까지 휴가 신청서를 써서 분대장을 거쳐 중대장에게 제출합니다. 같은 기간 신청자가 많으면 중대장이 조정합니다.",
     ["휴가는 언제 나갈 수 있나요", "휴가 신청은 어떻게 하나요", "휴가 며칠 전에 신청해야 하나요", "휴가 신청서는 누구에게 내나요",
      "첫 휴가 때 뭘 준비해야 해요", "휴가 인원이 몰리면 어떻게 정해요"]),
    ("outing", "외출·외박", "당직 계통",
     "외출은 주말과 공휴일 09시부터 21시까지입니다. 나가기 전 행선지와 연락처를 당직 계통에 보고합니다.",
     ["외출은 언제 할 수 있나요", "주말에 외출 몇 시까지예요", "외출 나가려면 뭘 보고해야 하나요", "외박은 어떻게 신청해요",
      "공휴일에도 외출 되나요", "외출 때 어디 간다고 말해야 해요"]),
    ("late", "복귀 지연", "당직사관",
     "부득이하게 복귀가 늦어질 것 같으면 복귀 예정 시각 전에 당직사관에게 연락하고 사유를 보고합니다.",
     ["복귀가 늦어질 것 같아요", "차가 막혀서 늦게 들어가면 어떡해요", "복귀 시간에 못 맞추면 누구한테 연락하나요", "늦게 복귀하면 처벌받나요",
      "귀대 시간이 지날 것 같아요", "버스를 놓쳐서 복귀가 늦어요"]),
    ("visit", "면회", "중대 행정반",
     "면회는 주말과 공휴일 오후에 면회실에서 할 수 있습니다. 가족이 방문하기 전 분대장을 통해 면회 신청을 해 두세요.",
     ["가족이 면회 오려면 어떻게 해요", "면회는 언제 가능한가요", "면회 신청은 어디서 하나요", "부모님이 오시는데 면회 되나요",
      "면회 장소가 어디예요", "친구도 면회 올 수 있나요"]),
    ("phone", "휴대전화·전화", "중대 행정반",
     "휴대전화는 평일 18시부터 21시, 주말과 공휴일 09시부터 21시에 사용할 수 있습니다. 사용 시간 외에는 지정 장소에 보관합니다.",
     ["휴대전화는 언제 쓸 수 있나요", "핸드폰 사용 시간이 어떻게 되나요", "가족에게 전화하고 싶은데 언제 가능해요", "폰 반납은 언제 해요",
      "평일에도 전화할 수 있나요", "휴대폰 보관은 어디에 하나요"]),
    ("welfare", "복지시설", "복지 담당",
     "PX와 체력단련실은 개인정비 시간과 주말에 이용합니다. 시설 소독·점검 시간에는 이용이 제한됩니다.",
     ["PX는 몇 시에 여나요", "체력단련실은 언제 쓸 수 있어요", "복지시설 이용 시간이 궁금해요", "PX에서 뭘 살 수 있나요",
      "운동하려면 어디로 가야 해요", "도서관은 언제 열어요"]),
    ("meal", "식사", "취사 담당",
     "식사는 아침 07시, 점심 12시, 저녁 18시에 식당에서 합니다. 근무나 훈련으로 시간을 맞출 수 없으면 분대장에게 알려 별도 배식을 받습니다.",
     ["식사 시간은 언제예요", "아침 밥은 몇 시에 먹나요", "근무 때문에 식사를 못 하면 어떻게 해요", "식당 위치가 어디예요",
      "저녁 식사는 몇 시부터인가요", "점심은 몇 시에 나와요"]),
    ("medical", "진료·의무", "의무실",
     "몸이 아프면 분대장에게 알리고 의무실에서 진료를 받습니다. 부상자나 환자가 생기면 즉시 당직 계통과 의무실에 알립니다.",
     ["몸이 아픈데 어떻게 해요", "의무실은 어디에 있나요", "다쳤을 때는 어디에 말해요", "감기 걸리면 진료받을 수 있나요",
      "병원에 가야 할 것 같아요", "예방접종은 언제 하나요"]),
    ("security", "보안·촬영", "보안 담당",
     "부대 안에서 휴대전화로 시설·문서·화면을 촬영하거나 외부로 전송하는 일은 금지입니다. 부대 자료를 외부 AI 서비스에 입력하는 것도 안 됩니다.",
     ["부대에서 사진 찍어도 되나요", "생활관 모습을 SNS에 올려도 되나요", "부대 자료를 AI에 넣어도 되나요", "촬영하면 안 되는 곳이 있나요",
      "휴대폰 카메라 써도 돼요", "ChatGPT에 부대 문서를 입력해도 되나요"]),
    ("account", "계정·비밀번호", "정보보호 담당",
     "업무용 계정 비밀번호는 분기마다 바꾸고 영문·숫자·특수문자를 섞어 9자리 이상으로 만듭니다. 비밀번호는 남에게 알려 주지 않습니다.",
     ["비밀번호는 몇 자리로 만들어요", "비번 바꾸는 주기가 어떻게 돼요", "계정 비밀번호를 잊어버렸어요", "비밀번호 규칙이 뭐예요",
      "전우에게 내 비번을 알려줘도 되나요", "업무 계정은 어떻게 받아요"]),
    ("selfdev", "자기개발", "정훈교육 담당",
     "자격증 응시료와 온라인 강좌 수강료는 연간 한도 안에서 실비로 지원합니다. 신청할 때 증빙 서류를 함께 냅니다.",
     ["자격증 시험 응시료 지원되나요", "온라인 강의 들을 수 있나요", "자기개발 지원은 어떻게 신청해요", "한국사 시험 보고 싶어요",
      "수강료도 지원해 주나요", "자격증 공부는 언제 할 수 있어요"]),
    ("counsel", "고충·상담", "병영생활상담관",
     "고충이나 건의는 소원수리함, 중대장 면담, 상담관 상담으로 말할 수 있고 이로 인한 불이익은 없습니다.",
     ["힘든 일이 있으면 어디에 말해요", "상담받고 싶은데 어떻게 해요", "선임과 문제가 생겼어요", "건의사항은 어디에 내요",
      "소원수리함이 어디 있어요", "고민을 털어놓을 사람이 필요해요"]),
    ("room", "생활관·청소", "분대장",
     "생활관 청소는 매일 아침 점호 후 20분간 합니다. 개인 전열기는 사용할 수 없고, 난방기 이상은 행정반에 신고합니다.",
     ["청소는 언제 하나요", "생활관에서 전기난로 써도 되나요", "청소 구역은 누가 정해요", "난방기가 고장 났어요",
      "생활관 정리 기준이 뭐예요", "방 청소 몇 분 동안 해요"]),
    ("supply", "보급품·세탁", "군수 담당",
     "피복과 개인 보급품은 전입 첫 주에 군수 담당에게 수령하고, 사이즈가 맞지 않으면 분대장을 통해 교환을 신청합니다. 세탁은 지정된 세탁 시간에 합니다.",
     ["군복 사이즈가 안 맞아요", "보급품은 어디서 받아요", "빨래는 어떻게 해요", "전투화를 교환하고 싶어요",
      "세탁실은 언제 쓸 수 있나요", "속옷이나 양말도 보급되나요"]),
]
INTENT_IDS = [f[0] for f in FAQ]
INTENT_NAME = {f[0]: f[1] for f in FAQ}
DEPT = {f[0]: f[2] for f in FAQ}
ANSWER = {f[0]: f[3] for f in FAQ}
TRAIN = [(q, f[0]) for f in FAQ for q in f[4][:4]]   # 60개
TEST = [(q, f[0]) for f in FAQ for q in f[4][4:]]    # 30개

# ---- 공통: 질문 정리, 규칙 기반 분류기, TF-IDF 분류기, 평가 ----
STOPWORDS = ["언제까지", "몇 시에", "어디서", "어디에", "어떻게", "어떡해요", "알려주세요", "궁금해요", "되나요", "하나요", "있나요",
             "언제", "어디", "몇 시", "돼요", "해요", "뭐예요", "수 있", "같아요", "주세요", "해야"]


def normalize(q):
    """웃음·특수문자 제거, 문장 끝 어미와 질문 투 말(언제·어떻게 등) 제거"""
    q = re.sub(r"[ㅋㅎㅠㅜ]+|[!?~.,]+", " ", q)
    for w in sorted(STOPWORDS, key=len, reverse=True):
        q = q.replace(w, " ")
    return re.sub(r"\s+", " ", q).strip()


RULES = {   # [교육용가상자료] 의도별 핵심 키워드 (Baseline)
    "schedule": ["일과", "점호", "취침", "시간"], "leave": ["휴가"], "outing": ["외출", "외박"],
    "late": ["복귀", "늦"], "visit": ["면회"], "phone": ["휴대전화", "핸드폰", "전화"],
    "welfare": ["PX", "체력단련"], "meal": ["식사", "식당"], "medical": ["아프", "의무실"],
    "security": ["촬영", "사진"], "account": ["비밀번호", "비번"], "selfdev": ["자격증", "응시료"],
    "counsel": ["상담", "고충"], "room": ["청소", "난방"], "supply": ["군복", "빨래"],
}


def rule_classify(q):
    best, best_n = None, 0
    for intent, kws in RULES.items():
        n = sum(k in q for k in kws)
        if n > best_n:
            best, best_n = intent, n
    return best          # 맞는 키워드가 없으면 None


class TfidfClassifier:
    """의도마다 '이름 + 답변 + 학습 질문 4개'를 한 문서로 묶고, 질문과 가장 비슷한 의도 문서를 찾는다."""
    def __init__(self, faq=None, n_train=4):
        faq = faq or FAQ
        self.ids = [f[0] for f in faq]
        self.docs = [normalize(f[1] + " " + f[3] + " " + " ".join(f[4][:n_train])) for f in faq]
        self.vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3), sublinear_tf=True).fit(self.docs)
        self.D = self.vec.transform(self.docs)

    def scores(self, q):
        s = cosine_similarity(self.vec.transform([normalize(q)]), self.D).ravel()
        return dict(zip(self.ids, (float(x) for x in s)))

    def top(self, q, k=2):
        return sorted(self.scores(q).items(), key=lambda x: -x[1])[:k]

    def predict(self, q):
        return self.top(q, 1)[0]


def accuracy(pred, gold):
    return float(np.mean([p == g for p, g in zip(pred, gold)]))

# ---- 공통: 대화 흐름 (인사 -> 의도 분류 -> 되묻기/모름 안내/답변) ----
GREET = ["안녕", "안녕하세요", "반갑", "처음 왔", "전입 왔", "하이"]
THANKS = ["감사", "고마워", "고맙", "수고"]
BYE = ["종료", "그만", "잘 가", "다음에"]
THRESH, MARGIN = 0.15, 0.04      # 최소 유사도, 1·2위 점수 차이 기준
GENERAL_DEPT = "중대 행정반 또는 분대장"


def decide(clf, q, thresh=THRESH, margin=MARGIN):
    """('greet'|'thanks'|'bye'|'answer'|'clarify'|'unknown', 의도, 점수, 후보)"""
    if any(w in q for w in BYE):
        return "bye", None, 0.0, []
    if any(w in q for w in THANKS):
        return "thanks", None, 0.0, []
    top = clf.top(q, 2)
    (i1, s1), (i2, s2) = top[0], top[1]
    if any(w in q for w in GREET) and s1 < 0.25:
        return "greet", None, s1, top
    if s1 < thresh:
        return "unknown", None, s1, top
    if s1 - s2 < margin:
        return "clarify", i1, s1, top
    return "answer", i1, s1, top

RULES_TEXT = [
    "아래 [FAQ]에 있는 내용만 근거로 답한다. FAQ에 없으면 '제가 가진 안내에는 없습니다'라고 말하고 담당 부서 확인을 권한다.",
    "시간·기한·숫자를 FAQ에 없는 값으로 지어내지 않는다.",
    "질문이 두 가지 이상으로 해석되면 먼저 어떤 내용인지 되묻는다.",
    "답변은 3문장 이내의 존댓말로 쓰고, 마지막에 확인할 담당 부서를 알려 준다.",
    "실제 부대 정보, 개인정보, 비밀 사항은 묻지도 답하지도 않는다. 모든 내용은 교육용 가상 자료다.",
]


def build_system_prompt(faq):
    lines = ["[역할] 당신은 가상 한빛대대의 전입 병사 생활 안내 챗봇이다. 말투는 친절한 존댓말이다.", "", "[규칙]"]
    lines += [f"{i}. {r}" for i, r in enumerate(RULES_TEXT, 1)]
    lines += ["", "[FAQ]"]
    for code, name, dept, answer, examples in faq:
        lines.append(f"- 주제: {name} / 담당: {dept}\n  답변: {answer}\n  자주 나오는 질문: {', '.join(examples[:3])}")
    lines += ["", "[모를 때 문구] '질문에 맞는 안내를 찾지 못했습니다. " + GENERAL_DEPT + "에게 확인해 주세요.'"]
    return "\n".join(lines)


def build_user_message(clf, q):
    kind, intent, score, top = decide(clf, q)
    hint = f"(분류기 힌트: {INTENT_NAME[top[0][0]]}, 유사도 {top[0][1]:.2f})" if kind in ("answer", "clarify") else "(분류기 힌트: 해당 주제 없음)"
    return f"[병사 질문] {q}\n{hint}"


def check_prompt(p):
    return {
        "역할 지정": "[역할]" in p,
        "규칙 5개": all(f"{i}." in p for i in range(1, 6)),
        "FAQ 15개 주제": p.count("- 주제:") == len(FAQ),
        "모든 답변 포함": all(f[3] in p for f in FAQ),
        "모를 때 문구": "[모를 때 문구]" in p,
        "담당 부서 안내 규칙": "담당 부서" in p,
    }


sp = build_system_prompt(FAQ)
print(sp[:1100] + "\n   ... (중략) ...\n")
print("[사용자 메시지 예]")
clf = TfidfClassifier()
for q in ["휴가 신청 언제까지 해야 돼요", "사단장 방문 일정은?"]:
    print(build_user_message(clf, q), "\n")
res = check_prompt(sp)
for k, v in res.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")
print(f"점검 {sum(res.values())}/{len(res)} 통과, 프롬프트 길이 {len(sp)}자")
print("이 프롬프트를 공개 생성형 AI에 넣어 쓸 때도 가상 자료만 사용하고, 실제 부대 정보는 입력하지 않는다.")


[역할] 당신은 가상 한빛대대의 전입 병사 생활 안내 챗봇이다. 말투는 친절한 존댓말이다.

[규칙]
1. 아래 [FAQ]에 있는 내용만 근거로 답한다. FAQ에 없으면 '제가 가진 안내에는 없습니다'라고 말하고 담당 부서 확인을 권한다.
2. 시간·기한·숫자를 FAQ에 없는 값으로 지어내지 않는다.
3. 질문이 두 가지 이상으로 해석되면 먼저 어떤 내용인지 되묻는다.
4. 답변은 3문장 이내의 존댓말로 쓰고, 마지막에 확인할 담당 부서를 알려 준다.
5. 실제 부대 정보, 개인정보, 비밀 사항은 묻지도 답하지도 않는다. 모든 내용은 교육용 가상 자료다.

[FAQ]
- 주제: 일과 시간 / 담당: 중대 행정반
  답변: 기상은 06시 30분, 점호는 21시 30분, 취침은 22시입니다. 평일 18시부터 21시까지는 개인정비 시간입니다.
  자주 나오는 질문: 하루 일과는 어떻게 되나요, 기상 시간이 몇 시예요, 점호는 몇 시에 하나요
- 주제: 휴가 / 담당: 중대 행정반
  답변: 정기 휴가는 시행 7일 전까지 휴가 신청서를 써서 분대장을 거쳐 중대장에게 제출합니다. 같은 기간 신청자가 많으면 중대장이 조정합니다.
  자주 나오는 질문: 휴가는 언제 나갈 수 있나요, 휴가 신청은 어떻게 하나요, 휴가 며칠 전에 신청해야 하나요
- 주제: 외출·외박 / 담당: 당직 계통
  답변: 외출은 주말과 공휴일 09시부터 21시까지입니다. 나가기 전 행선지와 연락처를 당직 계통에 보고합니다.
  자주 나오는 질문: 외출은 언제 할 수 있나요, 주말에 외출 몇 시까지예요, 외출 나가려면 뭘 보고해야 하나요
- 주제: 복귀 지연 / 담당: 당직사관
  답변: 부득이하게 복귀가 늦어질 것 같으면 복귀 예정 시각 전에 당직사관에게 연락하고 사유를 보고합니다.
  자주 나오는 질문: 복귀가 늦어질 것 같아요, 차가 막혀서 늦게 들어가면 어떡해요, 복귀 시간에 못 맞추면 누구한테 연락하나요
- 주제: 면회 / 담당: 중대 행정반
  답변: 면회는 주말과 공휴일 오후에 면회실에서